# Age-adjusted versus unadjusted gait classification

This participant-level baseline asks whether age improves stroke classification because it carries genuine gait information or because it acts as a demographic shortcut. Every age residualization step is fitted inside the training fold only. This is a shallow audit of the primary task, not a replacement for the pooled deep-learning benchmark.

In [1]:
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src')); PROCESSED = PROJECT_ROOT / 'data' / 'processed'; INTERIM = PROJECT_ROOT / 'data' / 'interim'
from features import voisard
FEATURES = ['cadence_steps_per_min', 'stride_time_mean_s', 'stride_time_cv_mean', 'lb_accel_rms', 'foot_accel_rms_mean', 'he_accel_rms']
manifest = pd.read_csv(INTERIM / 'ml_readiness_manifest.csv'); age_map = manifest[manifest.dataset_id.eq('voisard_2025')].groupby('subject').age.first().astype(float).to_dict()
trial_features = voisard.build_feature_table(); participant = trial_features.groupby(['subject', 'label'], as_index=False)[FEATURES].mean(); participant['age'] = participant.subject.map(age_map); participant['stroke'] = participant.label.eq('CVA').astype(int); participant = participant.dropna(subset=FEATURES + ['age']).reset_index(drop=True)
print('Participants:', len(participant), '| healthy:', int((participant.stroke == 0).sum()), '| stroke:', int((participant.stroke == 1).sum()))

Participants: 122 | healthy: 73 | stroke: 49


In [2]:
def residualize(train, test):
    train_out = train[FEATURES].copy(); test_out = test[FEATURES].copy(); age_train = train.age.to_numpy(float); age_test = test.age.to_numpy(float); design = np.column_stack([np.ones(len(train)), age_train - age_train.mean()]); design_test = np.column_stack([np.ones(len(test)), age_test - age_train.mean()])
    for feature in FEATURES:
        beta, _, _, _ = np.linalg.lstsq(design, train[feature].to_numpy(float), rcond=None); train_out[feature] = train[feature].to_numpy(float) - design @ beta; test_out[feature] = test[feature].to_numpy(float) - design_test @ beta
    return train_out, test_out

def fit_eval(train_x, test_x, train_y, test_y):
    scaler = StandardScaler().fit(train_x); model = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0).fit(scaler.transform(train_x), train_y); probability = model.predict_proba(scaler.transform(test_x))[:, 1]; return {'roc_auc': roc_auc_score(test_y, probability), 'balanced_accuracy': balanced_accuracy_score(test_y, probability >= 0.5)}

rows = []
for seed in [42, 52, 62]:
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for fold, (train_pos, test_pos) in enumerate(splitter.split(participant, participant.stroke)):
        train = participant.iloc[train_pos]; test = participant.iloc[test_pos]; y_train = train.stroke.to_numpy(); y_test = test.stroke.to_numpy()
        raw_train, raw_test = train[FEATURES], test[FEATURES]; adj_train, adj_test = residualize(train, test)
        for model, train_x, test_x in [('gait_raw', raw_train, raw_test), ('gait_age_residualized', adj_train, adj_test), ('gait_plus_age', train[FEATURES + ['age']], test[FEATURES + ['age']]), ('age_only', train[['age']], test[['age']])]:
            row = fit_eval(train_x, test_x, y_train, y_test); row.update({'model': model, 'seed': seed, 'fold': fold, 'participants': len(test)}); rows.append(row)
results = pd.DataFrame(rows); print(results.groupby('model')[['roc_auc', 'balanced_accuracy']].agg(['mean', 'std']).round(3).to_string())

                      roc_auc        balanced_accuracy       
                         mean    std              mean    std
model                                                        
age_only                0.803  0.103             0.788  0.067
gait_age_residualized   0.882  0.057             0.786  0.101
gait_plus_age           0.968  0.040             0.923  0.052
gait_raw                0.973  0.030             0.921  0.052


In [3]:
results.to_csv(PROCESSED / 'age_adjusted_gait_baseline_metrics.csv', index=False); print('Saved age-adjusted gait baseline outputs.')

Saved age-adjusted gait baseline outputs.


## Interpretation gate

A gait-plus-age improvement is not automatically desirable because the primary Voisard task has an age imbalance and Felius has no public age metadata. Age should only enter a future model if the improvement survives age-matched external validation and remains clinically appropriate.